In [26]:
import math
import random

from dataclasses import dataclass


# =====================================================
# CONFIG
# =====================================================

CAPACITY = 300
BUFFER_CAPACITY = 40
INITIAL_C0 = 200

ALPHA = 0.3

LAMBDA_1 = 3.0   # буфер
LAMBDA_2 = 5.0   # остатки
LAMBDA_3 = 5.0   # необслуженные
LAMBDA_4 = 1.0   # доготовка



# =====================================================
# REAL SCHOOL DATA
# =====================================================

TIME_PERIODS = [

    # ---------------------------------------------
    # ПРИЁМ ПИЩИ 1
    # ---------------------------------------------

    {
        "period": 0,
        "meal_type": "meal_1",
        "groups": "1, 2 классы",
        "students_total": 100,
        "benefit_students": 100,
    },

    {
        "period": 1,
        "meal_type": "meal_1",
        "groups": "3, 4, 5 классы",
        "students_total": 150,
        "benefit_students": 128,
    },

    {
        "period": 2,
        "meal_type": "meal_1",
        "groups": "6, 7, 8, 9 классы",
        "students_total": 200,
        "benefit_students": 112,
    },

    # ---------------------------------------------
    # ПРИЁМ ПИЩИ 2
    # ---------------------------------------------

    {
        "period": 3,
        "meal_type": "meal_2",
        "groups": "1 класс",
        "students_total": 50,
        "benefit_students": 28,
    },

    {
        "period": 4,
        "meal_type": "meal_2",
        "groups": "2, 3 класс",
        "students_total": 100,
        "benefit_students": 56,
    },

    {
        "period": 5,
        "meal_type": "meal_2",
        "groups": "4, 5, 6, 8 классы",
        "students_total": 200,
        "benefit_students": 112,
    },

    {
        "period": 6,
        "meal_type": "meal_2",
        "groups": "7, 9 классы",
        "students_total": 100,
        "benefit_students": 56,
    },

    {
        "period": 7,
        "meal_type": "meal_2",
        "groups": "школа полного дня",
        "students_total": 200,
        "benefit_students": 112,
    },
]


# =====================================================
# STATE
# =====================================================

@dataclass
class State:
    period: int
    b: int
    lambda_p: float
    c0: int
    buffer: int


# =====================================================
# ACTIONS
# =====================================================

@dataclass
class MainAction:
    prep_b: int
    prep_p: int


@dataclass
class BufferAction:
    cook_b: int
    cook_p: int


# =====================================================
# DEMAND MODELS
# =====================================================

def sample_absences(b_i):

    d_max = max(1, int(0.15 * b_i))

    weights = []

    for d in range(d_max + 1):
        weight = (d_max - d + 1) ** 2
        weights.append(weight)

    return random.choices(
        population=list(range(d_max + 1)),
        weights=weights,
        k=1,
    )[0]


def poisson_sample(lmbd):

    L = math.exp(-lmbd)

    k = 0
    p = 1

    while p > L:
        k += 1
        p *= random.random()

    return k - 1


def sample_paid(lambda_i):
    return poisson_sample(lambda_i)


# =====================================================
# UPDATE LAMBDA
# =====================================================

def update_lambda(previous_lambda, actual_paid):

    return (
        ALPHA * actual_paid
        + (1 - ALPHA) * previous_lambda
    )


# =====================================================
# BUFFER UPDATE
# =====================================================

def update_buffer(
    current_buffer,
    planned,
    actual,
    cooked,
    same_meal_type,
):

    shortage = max(0, actual - planned)

    used_buffer = min(current_buffer, shortage)

    leftovers = max(0, planned - actual)

    buffer_after = (
        current_buffer
        - used_buffer
        + leftovers
    )

    if same_meal_type:
        next_buffer = buffer_after + cooked
    else:
        next_buffer = cooked

    return (
        next_buffer,
        used_buffer,
        leftovers,
    )


# =====================================================
# SIMPLE POLICY
# =====================================================

def simple_policy(state):

    main_action = MainAction(

        # готовим всех льготников
        prep_b=state.b,

        # готовим по прогнозу платников
        prep_p=round(state.lambda_p),
    )

    # фиксированная доготовка
    buffer_action = BufferAction(
        cook_b=10,
        cook_p=10,
    )

    return main_action, buffer_action


# =====================================================
# ONE PERIOD SIMULATION
# =====================================================

def simulate_period(state, policy):

    main_action, buffer_action = policy(state)

    # -------------------------------------------------
    # СЛУЧАЙНЫЙ СПРОС
    # -------------------------------------------------

    d = sample_absences(state.b)

    w = sample_paid(state.lambda_p)

    actual_benefit = state.b - d

    actual = actual_benefit + w

    planned = (
        main_action.prep_b
        + main_action.prep_p
    )

    cooked = (
        buffer_action.cook_b
        + buffer_action.cook_p
    )

    # -------------------------------------------------
    # ОБСЛУЖИВАНИЕ
    # -------------------------------------------------

    served = min(
        actual,
        planned + state.buffer,
    )

    unserved = max(
        0,
        actual - (planned + state.buffer),
    )

    # -------------------------------------------------
    # СЛЕДУЮЩИЙ ПЕРИОД
    # -------------------------------------------------

    current_period = state.period

    next_period = current_period + 1

    if next_period >= len(TIME_PERIODS):
        next_period = 0

    same_meal_type = (
        TIME_PERIODS[current_period]["meal_type"]
        ==
        TIME_PERIODS[next_period]["meal_type"]
    )

    # -------------------------------------------------
    # ОБНОВЛЕНИЕ БУФЕРА
    # -------------------------------------------------

    (
        next_buffer,
        used_buffer,
        leftovers,
    ) = update_buffer(
        current_buffer=state.buffer,
        planned=planned,
        actual=actual,
        cooked=cooked,
        same_meal_type=same_meal_type,
    )

    # -------------------------------------------------
    # ОБНОВЛЕНИЕ LAMBDA
    # -------------------------------------------------

    next_lambda = update_lambda(
        previous_lambda=state.lambda_p,
        actual_paid=w,
    )

    # -------------------------------------------------
    # ОБНОВЛЕНИЕ C0
    # -------------------------------------------------

    next_c0 = max(
        0,
        state.c0 - cooked,
    )

    # -------------------------------------------------
    # НОВОЕ СОСТОЯНИЕ
    # -------------------------------------------------

    next_state = State(
        period=next_period,

        b=TIME_PERIODS[next_period][
            "benefit_students"
        ],

        lambda_p=round(next_lambda, 2),

        c0=next_c0,

        buffer=next_buffer,
    )


    # =========================
    # COST FUNCTION (g)
    # =========================

    buffer_cost = 0  # λ1 * used_buffer

    leftover_cost = 0  # λ2 * leftovers

    unserved_cost = (unserved ** 2)  # λ3 * L_i^2

    cooking_cost = cooked  # λ4 * tilde A_i

    total_cost = (
    LAMBDA_1 * used_buffer
    + LAMBDA_2 * leftovers
    + LAMBDA_3 * (unserved ** 2)
    + LAMBDA_4 * cooked
)

    # -------------------------------------------------
    # LOGGING
    # -------------------------------------------------

    result = {

        "groups":
            TIME_PERIODS[current_period]["groups"],

        "meal_type":
            TIME_PERIODS[current_period]["meal_type"],

        "students_total":
            TIME_PERIODS[current_period][
                "students_total"
            ],

        "state":
            state,

        "main_action":
            main_action,

        "buffer_action":
            buffer_action,

        "absent_benefit":
            d,

        "actual_benefit":
            actual_benefit,

        "paid_students":
            w,

        "actual_demand":
            actual,

        "planned":
            planned,

        "served":
            served,

        "unserved":
            unserved,

        "used_buffer":
            used_buffer,

        "leftovers":
            leftovers,

        "next_state":
            next_state,

        "cost": total_cost,
    }

    return next_state, result


# =====================================================
# ONE DAY SIMULATION
# =====================================================

def simulate_day(initial_state, policy):

    state = initial_state
    logs = []

    cumulative_cost = 0  # 👈 ДОБАВИЛИ

    for _ in range(len(TIME_PERIODS)):

        state, result = simulate_period(state, policy)

        cumulative_cost += result["cost"]  # 👈 НАКАПЛИВАЕМ

        result["cumulative_cost"] = cumulative_cost  # 👈 ДОБАВЛЯЕМ В ЛОГ

        logs.append(result)

    return logs


# =====================================================
# MAIN
# =====================================================

def main():

    initial_state = State(

        period=0,

        b=TIME_PERIODS[0][
            "benefit_students"
        ],

        lambda_p=20,

        c0=INITIAL_C0,

        buffer=20,
    )

    logs = simulate_day(
        initial_state,
        simple_policy,
    )

    for i, log in enumerate(logs):

        print("=" * 70)

        print(
            f"PERIOD {i}"
        )

        print(
            f"MEAL TYPE: "
            f"{log['meal_type']}"
        )

        print(
            f"GROUPS: "
            f"{log['groups']}"
        )

        print(
            f"TOTAL STUDENTS: "
            f"{log['students_total']}"
        )

        print(
            f"BENEFIT EXPECTED: "
            f"{log['state'].b}"
        )

        print(
            f"LAMBDA PAID: "
            f"{log['state'].lambda_p}"
        )

        print(
            f"ABSENT BENEFIT: "
            f"{log['absent_benefit']}"
        )

        print(
            f"ACTUAL BENEFIT: "
            f"{log['actual_benefit']}"
        )

        print(
            f"PAID STUDENTS: "
            f"{log['paid_students']}"
        )

        print(
            f"ACTUAL DEMAND: "
            f"{log['actual_demand']}"
        )

        print(
            f"PLANNED PORTIONS: "
            f"{log['planned']}"
        )

        print(
            f"SERVED: "
            f"{log['served']}"
        )

        print(
            f"UNSERVED: "
            f"{log['unserved']}"
        )

        print(
            f"USED BUFFER: "
            f"{log['used_buffer']}"
        )

        print(
            f"LEFTOVERS: "
            f"{log['leftovers']}"
        )

        print(
            f"NEXT STATE: "
            f"{log['next_state']}"
        )
        print(f"PERIOD COST: {log['cost']}")
        print(f"CUMULATIVE COST: {log['cumulative_cost']}")
        

    print("=" * 70)
    print(f"TOTAL COST FOR THE DAY: {logs[-1]['cumulative_cost']}")


if __name__ == "__main__":
    main()

PERIOD 0
MEAL TYPE: meal_1
GROUPS: 1, 2 классы
TOTAL STUDENTS: 100
BENEFIT EXPECTED: 100
LAMBDA PAID: 20
ABSENT BENEFIT: 2
ACTUAL BENEFIT: 98
PAID STUDENTS: 29
ACTUAL DEMAND: 127
PLANNED PORTIONS: 120
SERVED: 127
UNSERVED: 0
USED BUFFER: 7
LEFTOVERS: 0
NEXT STATE: State(period=1, b=128, lambda_p=22.7, c0=180, buffer=33)
PERIOD COST: 41.0
CUMULATIVE COST: 41.0
PERIOD 1
MEAL TYPE: meal_1
GROUPS: 3, 4, 5 классы
TOTAL STUDENTS: 150
BENEFIT EXPECTED: 128
LAMBDA PAID: 22.7
ABSENT BENEFIT: 0
ACTUAL BENEFIT: 128
PAID STUDENTS: 25
ACTUAL DEMAND: 153
PLANNED PORTIONS: 151
SERVED: 153
UNSERVED: 0
USED BUFFER: 2
LEFTOVERS: 0
NEXT STATE: State(period=2, b=112, lambda_p=23.39, c0=160, buffer=51)
PERIOD COST: 26.0
CUMULATIVE COST: 67.0
PERIOD 2
MEAL TYPE: meal_1
GROUPS: 6, 7, 8, 9 классы
TOTAL STUDENTS: 200
BENEFIT EXPECTED: 112
LAMBDA PAID: 23.39
ABSENT BENEFIT: 3
ACTUAL BENEFIT: 109
PAID STUDENTS: 24
ACTUAL DEMAND: 133
PLANNED PORTIONS: 135
SERVED: 133
UNSERVED: 0
USED BUFFER: 0
LEFTOVERS: 2
NEXT S